# 05 — Gradio Demo

**用途**:啟動一個網頁 demo,上傳影片 → 標註影片(骨架 + track id + 狀態 + ALARM
橫幅)+ 偵測到的跌倒事件表 + `events.json` 下載。用來錄製 README 的兩張示範 GIF:
一張跌倒正確觸發、一張日常動作(ADL)正確不觸發。

**執行方式**:`Runtime → Run all`(GPU runtime,T4 即可;沒 GPU 也能跑,只是
姿態抽取較慢)。**最後一格會持續執行、不會結束**——這是預期行為:
`gr.Blocks.launch()` 會啟動一個網頁伺服器並印出一個公開連結
(`https://xxxx.gradio.live`,`share=True` 有效期約 72 小時),notebook 就停在
那格等你操作,不是卡住;要停止就手動中斷該格執行。

**操作方式**:
1. 點印出的 `https://xxxx.gradio.live` 連結(手機/另一台電腦也能開)。
2. 左側範例區已放好一支真實跌倒影片(`fall-01`)與一支日常動作影片
   (`adl-01`),點一下自動填入上傳欄;也可以自己上傳任意短片。
3. 按「開始偵測」,右側會依序出現:標註影片、事件表、`events.json` 下載連結。
4. 螢幕錄影 `fall-01`(展示正確觸發 ALARM)與 `adl-01`(展示正確不觸發),
   剪成兩支 <5MB 的 GIF 放進 `assets/`(可用 ffmpeg palettegen 壓,見 plan.md),
   README 會並排引用。

**為什麼是 fall-01 / adl-01,不是特別去挑的「最佳案例」**:兩支都是
[eval/metrics.json](../eval/metrics.json) 記錄在案的乾淨案例——`fall-01` 不在
test split 的 FN(漏報)清單、`adl-01` 不在 FP(誤報)清單,只是各自清單裡
「排序第一個」的乾淨案例,不是為了展示效果特別挑的極端好案例。系統真實的
失敗模式(1 個漏報 + 2 個誤報)已在 README「失敗分析」章節誠實揭露,這裡的
GIF 展示的是**典型**表現,不是掩蓋弱點。

In [ ]:
!nvidia-smi


In [ ]:
import os
if os.path.basename(os.getcwd()) != 'fall-detection-pose':
    if not os.path.exists('fall-detection-pose'):
        !git clone -q https://github.com/tun0000/fall-detection-pose.git
    %cd fall-detection-pose
!pip -q install -e ".[infer,demo]" pytest

# 同前幾本 notebook 的坑:pip install -e 之後 site.py 不會自動重新掃描 .pth,
# 這裡直接跑會 ModuleNotFoundError。reload(site) + 手動加 src/ 到 sys.path 雙重保險。
import importlib
import site
import sys

importlib.reload(site)
sys.path.insert(0, os.path.abspath("src"))

import fall_detection
print('fall_detection', fall_detection.__version__)

In [ ]:
# 規則引擎 + gradio_app 純函式單元測試(純 CPU,~10 秒):必須全綠
!python -m pytest -q


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/fall-detection-pose'
DATA_DIR = f'{DRIVE_ROOT}/data/urfd'
VIDEO_DIR = f'{DATA_DIR}/videos'

# fall-01 / adl-01:eval/metrics.json 記錄在案的乾淨案例(fall-01 不在 test split
# 的 FN 清單、adl-01 不在 FP 清單)——清單裡排序第一個乾淨案例,不是刻意挑的
# 最佳案例。找不到就印警告但不中斷:demo 一樣能用,只是範例區是空的,
# 使用者仍可自行上傳影片。
example_candidates = [f'{VIDEO_DIR}/fall-01.mp4', f'{VIDEO_DIR}/adl-01.mp4']
examples = [p for p in example_candidates if os.path.exists(p)]
missing = [p for p in example_candidates if p not in examples]
if missing:
    print(f'警告:找不到範例影片 {missing}(notebook 02 是否已跑過?),demo 仍可手動上傳使用')
print('範例影片:', examples)

In [ ]:
from fall_detection.app.gradio_app import build_demo

CONFIG_PATH = os.path.abspath('config.yaml')
demo = build_demo(config_path=CONFIG_PATH, example_videos=examples)

print('=== 啟動中,請等下面印出 https://xxxx.gradio.live 連結 ===')
demo.queue().launch(share=True, max_file_size='200mb')

## 錄完 GIF 之後

停止上面那格(左邊執行按鈕會變成方塊,點它中斷),把兩支螢幕錄影轉成 GIF
(建議 ffmpeg palettegen 兩階段壓縮到 <5MB、480p 寬),存成
`assets/demo_fall.gif` 與 `assets/demo_adl.gif`,告訴我檔案已經放好,
我會把它們接進 README。